# Plant Dataset — Decoy Strategy Comparison
Comparison of 4 decoy strategies (score_coord, score_coord_noise, nearest_neighbor, nearest_neighbor_noise) for accuracy estimation via Mix-Max FDR on the plant-daniella dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import tarfile
import io

print(f"NumPy {np.__version__}  PyTorch {torch.__version__}")


## Configuration

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

plt.rcParams.update({
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 9,
    'figure.titlesize': 15, 'font.family': 'serif',
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

os.makedirs('figures', exist_ok=True)


## Utility functions (calibration, baselines)

In [ ]:
def negentropy(logits):
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    probs = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
    return np.log(logits.shape[1]) - entropy

def calibration_temp(logits, labels, num_bins=15):
    if isinstance(logits, np.ndarray): logits = torch.from_numpy(logits).float()
    if isinstance(labels, np.ndarray): labels = torch.from_numpy(labels).long()
    temps = torch.linspace(0.1, 5.0, 50)
    best_temp, best_ece = 1.0, float('inf')
    for temp in temps:
        p = torch.softmax(logits / temp, dim=1)
        conf, pred = p.max(1); acc = (pred == labels).float()
        bins = torch.linspace(0, 1, num_bins + 1)
        ece = sum(
            ((conf > bins[i]) & (conf <= bins[i+1])).float().mean() *
            abs(conf[(conf > bins[i]) & (conf <= bins[i+1])].mean() -
                acc[(conf > bins[i]) & (conf <= bins[i+1])].mean()).item()
            for i in range(num_bins)
            if ((conf > bins[i]) & (conf <= bins[i+1])).sum() > 0
        )
        if ece < best_ece: best_ece, best_temp = ece, temp.item()
    return best_temp

def _to_tensor(x):
    return torch.from_numpy(x).float() if isinstance(x, np.ndarray) else x

def predict_ATC_maxconf(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = torch.softmax(src_logits, 1).amax(1)
    tgt_sc = torch.softmax(tgt_logits, 1).amax(1)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_ATC_negent(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = negentropy(src_logits); tgt_sc = negentropy(tgt_logits)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_AC(src_logits, src_labels, tgt_logits):
    return torch.softmax(_to_tensor(tgt_logits), 1).amax(1).mean().item()

def predict_DOC(src_logits, src_labels, tgt_logits):
    sl = _to_tensor(src_logits); sb = _to_tensor(src_labels).long(); tl = _to_tensor(tgt_logits)
    src_conf = torch.softmax(sl, 1).amax(1).mean().item()
    tgt_conf = torch.softmax(tl, 1).amax(1).mean().item()
    src_acc  = (sl.argmax(1) == sb).float().mean().item()
    return src_acc + (tgt_conf - src_conf)

BASELINE_METHODS = {'ATC': predict_ATC_maxconf, 'ATC-NE': predict_ATC_negent,
                    'AC': predict_AC, 'DOC': predict_DOC}

try:
    import ot
    def predict_COT(sl, sb, tl):
        sl = _to_tensor(sl); sb = _to_tensor(sb).long(); tl = _to_tensor(tl)
        nc = sl.shape[1]
        lbl_dist = F.one_hot(sb, nc).float().mean(0)
        tp = torch.softmax(tl, 1)
        cost = torch.stack([(tp - F.one_hot(torch.tensor(k), nc).float()).abs().sum(1) / 2 for k in range(nc)], 1)
        ot_plan = ot.emd(np.ones(len(tp)) / len(tp), lbl_dist.numpy(), cost.numpy())
        ot_cost = (ot_plan * cost.numpy()).sum()
        return 1 - (ot_cost + torch.softmax(sl, 1).amax(1).mean().item() - (sl.argmax(1) == sb).float().mean().item())
    BASELINE_METHODS['COT'] = predict_COT
    print("COT available")
except ImportError:
    print("COT unavailable (pip install POT)")
print(f"Baseline methods: {list(BASELINE_METHODS.keys())}")


## Flow model — building blocks

In [ ]:
class RobustFeatureNormalizer(nn.Module):
    def __init__(self, feature_dim, clip_val=5.0, momentum=0.01, eps=1e-6):
        super().__init__()
        self.clip_val = clip_val; self.momentum = momentum; self.eps = eps
        self.register_buffer('running_median', torch.zeros(feature_dim))
        self.register_buffer('running_iqr',    torch.ones(feature_dim))
        self.register_buffer('initialized',    torch.tensor(False))

    @torch.no_grad()
    def _update_stats(self, x):
        bm = x.median(0).values
        bi = (torch.quantile(x, 0.75, 0) - torch.quantile(x, 0.25, 0)).clamp(min=self.eps)
        if not self.initialized:
            self.running_median.copy_(bm); self.running_iqr.copy_(bi)
            self.initialized.fill_(True)
        else:
            self.running_median.mul_(1 - self.momentum).add_(bm * self.momentum)
            self.running_iqr.mul_(1 - self.momentum).add_(bi * self.momentum)

    def forward(self, x):
        if self.training: self._update_stats(x)
        if not self.initialized: return torch.tanh(x * 0.01)
        xn = ((x - self.running_median) / (self.running_iqr + self.eps)).clamp(-self.clip_val, self.clip_val)
        return torch.tanh(xn / self.clip_val)


class ActNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.log_scale = nn.Parameter(torch.zeros(dim))
        self.bias      = nn.Parameter(torch.zeros(dim))
        self.register_buffer('initialized', torch.tensor(False))

    def forward(self, x, reverse=False):
        if not self.initialized and not reverse:
            with torch.no_grad():
                self.bias.data      = -x.mean(0)
                self.log_scale.data = -x.std(0).clamp(min=1e-6).log()
            self.initialized.fill_(True)
        if not reverse:
            return (x + self.bias) * self.log_scale.exp(), self.log_scale.sum().expand(x.size(0))
        return x * (-self.log_scale).exp() - self.bias, -self.log_scale.sum().expand(x.size(0))


class CouplingLayer(nn.Module):
    def __init__(self, dim, feature_dim, hidden_dim=256, mask_type='first_half'):
        super().__init__()
        self.mask_type = mask_type
        self.d_in  = dim // 2 if mask_type == 'first_half' else dim - dim // 2
        self.d_out = dim - dim // 2 if mask_type == 'first_half' else dim // 2
        self.net = nn.Sequential(
            nn.Linear(self.d_in + feature_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU())
        self.scale_head     = nn.Sequential(nn.Linear(hidden_dim // 2, self.d_out), nn.Tanh())
        self.translate_head = nn.Linear(hidden_dim // 2, self.d_out)
        nn.init.zeros_(self.scale_head[0].weight); nn.init.zeros_(self.scale_head[0].bias)
        nn.init.zeros_(self.translate_head.weight); nn.init.zeros_(self.translate_head.bias)

    def _split(self, x):
        return (x[:, :self.d_in], x[:, self.d_in:]) if self.mask_type == 'first_half'                else (x[:, self.d_out:], x[:, :self.d_out])

    def _merge(self, x1, x2):
        return torch.cat([x1, x2], 1) if self.mask_type == 'first_half' else torch.cat([x2, x1], 1)

    def forward(self, x, features, reverse=False):
        x1, x2 = self._split(x); h = self.net(torch.cat([x1, features], 1))
        s, t = self.scale_head(h), self.translate_head(h)
        if not reverse:
            return self._merge(x1, x2 * torch.exp(s) + t), s.sum(1)
        return self._merge(x1, (x2 - t) * torch.exp(-s)), -s.sum(1)


## ScoreShiftFlow

In [ ]:
class ScoreShiftFlow(nn.Module):
    def __init__(self, score_dim=10, feature_dim=640, n_flows=12, hidden_dim=256,
                 encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.score_dim = score_dim
        self._log_2pi  = float(np.log(2 * np.pi))
        self.feature_norm    = RobustFeatureNormalizer(feature_dim, clip_val=clip_val, momentum=0.01)
        self.feature_encoder = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, encoder_dim), nn.LayerNorm(encoder_dim), nn.GELU())
        self.layers = nn.ModuleList()
        for i in range(n_flows):
            mask = 'first_half' if i % 2 == 0 else 'second_half'
            self.layers.append(CouplingLayer(score_dim, encoder_dim, hidden_dim, mask))
            if i < n_flows - 1:
                self.layers.append(ActNorm(score_dim))

    def encode(self, f): return self.feature_encoder(self.feature_norm(f))

    def forward(self, scores, features, reverse=False):
        enc = self.encode(features)
        ld  = torch.zeros(scores.size(0), device=scores.device)
        if not reverse:
            x = scores
            for layer in self.layers:
                x, d = layer(x, reverse=False) if isinstance(layer, ActNorm)                        else layer(x, enc, reverse=False)
                ld += d
            return x, ld
        z = scores
        for layer in reversed(self.layers):
            z, d = layer(z, reverse=True) if isinstance(layer, ActNorm)                    else layer(z, enc, reverse=True)
            ld += d
        return z, ld

    def log_prob(self, scores, features):
        z, ld = self.forward(scores, features)
        return -0.5 * (z ** 2).sum(1) - 0.5 * self.score_dim * self._log_2pi + ld

    def sample(self, features):
        z = torch.randn(features.size(0), self.score_dim, device=features.device)
        s, _ = self.forward(z, features, reverse=True)
        return s


## ScoreShiftFlowWrapper

In [ ]:
class ScoreShiftFlowWrapper(nn.Module):
    def __init__(self, num_classes=10, n_flows=12, feature_dim=640,
                 hidden_dim=256, encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.num_classes = num_classes
        self.flow = ScoreShiftFlow(num_classes, feature_dim, n_flows, hidden_dim, encoder_dim, clip_val)

    def train_flow(self, score_dataset, epochs=30, lr=3e-4, batch_size=256,
                   device='cuda', patience=5, grad_clip=1.0):
        self.flow.to(device).train()
        loader    = DataLoader(score_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        optimizer = torch.optim.AdamW(self.flow.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
        best_loss, best_state, no_improve = float('inf'), None, 0
        for epoch in range(epochs):
            total = 0.0; n = 0
            for _, feats, decoys, _ in loader:
                feats = feats.to(device); decoys = decoys.to(device)
                loss = -self.flow.log_prob(decoys, feats).mean()
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.flow.parameters(), grad_clip)
                optimizer.step(); total += loss.item(); n += 1
            scheduler.step(); avg = total / max(n, 1)
            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}  loss={avg:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
            if avg < best_loss - 1e-4:
                best_loss = avg; no_improve = 0
                best_state = {k: v.clone() for k, v in self.flow.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stop at epoch {epoch+1}")
                    self.flow.load_state_dict(best_state); break
        if best_state: self.flow.load_state_dict(best_state)
        print(f"  Done. Best loss: {best_loss:.4f}")
        return self

    def generate_decoys(self, score_dataset, device='cuda'):
        self.flow.to(device).eval()
        cnn_l, dc_l, lb_l = [], [], []
        loader = DataLoader(score_dataset, batch_size=256, shuffle=False, num_workers=0)
        with torch.no_grad():
            for cnn_sc, feats, _, labels in loader:
                feats = feats.to(device)
                dc_l.append(self.flow.sample(feats).cpu().numpy())
                cnn_l.append(cnn_sc.numpy()); lb_l.append(labels.numpy())
        return np.concatenate(cnn_l), np.concatenate(dc_l), np.concatenate(lb_l)


## ScoreFeatureDataset

In [ ]:
class ScoreFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, cnn_scores, features, target_decoy_scores, labels):
        self.cnn_scores          = cnn_scores
        self.features            = features
        self.target_decoy_scores = target_decoy_scores
        self.labels              = labels
    def __len__(self): return len(self.cnn_scores)
    def __getitem__(self, i):
        return self.cnn_scores[i], self.features[i], self.target_decoy_scores[i], self.labels[i]


## Pool building functions

In [ ]:
MIN_POOL = 30

def build_error_conditioned_pools(train_scores, train_labels, num_classes,
                                   verbose=True, max_pool_size=10_000):
    """pool_score[c] = score_c on examples where argmax=c AND label≠c (fallback: label≠c)."""
    rng = np.random.default_rng(0)
    pred_classes = train_scores.argmax(axis=1)
    pool_score = {}
    if verbose:
        print(f"Building pools  (train acc={(pred_classes == train_labels).mean():.4f})")
    for c in range(num_classes):
        err_mask = (pred_classes == c) & (train_labels != c)
        cands = train_scores[err_mask, c] if err_mask.sum() >= MIN_POOL                 else train_scores[train_labels != c, c]
        pool_score[c] = cands[rng.choice(len(cands), size=max_pool_size, replace=False)]                         if len(cands) > max_pool_size else cands
        if verbose:
            print(f"  class {c}: n={len(pool_score[c]):6d}  [{pool_score[c].min():.3f},{pool_score[c].max():.3f}]")
    return pool_score


def _build_decoy_score_coord(sc_np, pool_score, rng):
    """Replace coord c_hat by a random draw from pool_score[c_hat]."""
    pred = sc_np.argmax(axis=1); dc = sc_np.copy()
    for c in range(sc_np.shape[1]):
        mask = pred == c
        if mask.any() and len(pool_score.get(c, [])) > 0:
            dc[mask, c] = rng.choice(pool_score[c], size=mask.sum(), replace=True)
    return dc


## Data loading utilities

In [ ]:
def load_precomputed_data(path):
    """Load (features, logits, labels) from .pt or .tar.xz archive."""
    if path.endswith(('.tar.xz', '.tar.gz', '.tar.bz2', '.tar')):
        with tarfile.open(path, 'r:*') as tar:
            members = [m for m in tar.getmembers() if m.name.endswith('.pt')]
            if not members: raise FileNotFoundError(f"No .pt inside {path}")
            data = torch.load(io.BytesIO(tar.extractfile(members[0]).read()),
                              map_location='cpu', weights_only=False)
    else:
        data = torch.load(path, map_location='cpu', weights_only=False)
    print(list(data.keys()))
    feat_key  = 'hidden_features' if 'hidden_features' in data else 'test_hidden_features'
    logit_key = 'logits'          if 'logits'          in data else 'test_logits'
    label_key = 'labels'          if 'labels'          in data else 'test_labels'
    return data[feat_key].float(), data[logit_key].float(), data[label_key].long()


## Load precomputed plant dataset

In [ ]:
TRAIN_DATA_PATH = '/kaggle/input/datasets/arinaromashkina/plantlet/train_data2 (1).tar.xz'
TEST_DATA_PATH  = '/kaggle/input/datasets/arinaromashkina/plantlet/test_data2.tar.xz'

train_features, train_logits, train_labels_t = load_precomputed_data(TRAIN_DATA_PATH)
test_features,  test_logits,  test_labels_t  = load_precomputed_data(TEST_DATA_PATH)

train_scores_raw = train_logits.numpy()
train_labels_raw = train_labels_t.numpy()

NUM_CLASSES = train_logits.shape[1]
FEATURE_DIM = train_features.shape[1]

print(f"Train: logits={train_logits.shape}  features={train_features.shape}")
print(f"Test:  logits={test_logits.shape}   features={test_features.shape}")
print(f"NUM_CLASSES={NUM_CLASSES}  FEATURE_DIM={FEATURE_DIM}")
print(f"Train acc={(train_scores_raw.argmax(1) == train_labels_raw).mean():.4f}")


### Dataset statistics

In [ ]:
# ── True label distribution (train + test) ───────────────────────────────────
for split, labels in [('TRAIN', train_labels_raw), ('TEST', lb_np)]:
    counts = np.bincount(labels)
    nonzero = counts[counts > 0]
    top_cls = np.argsort(counts)[::-1][:10]
    print(f"\n{split}: {len(labels)} samples, {(counts > 0).sum()} classes with data")
    print(f"  samples/class — min: {nonzero.min()}  median: {int(np.median(nonzero))}  "
          f"max: {nonzero.max()}  mean: {nonzero.mean():.1f}")
    print(f"  Top 10 classes by true label count:")
    for c in top_cls:
        if counts[c] == 0: break
        print(f"    class {c:>4d}: {counts[c]:>5d}  ({counts[c]/len(labels)*100:.1f}%)")

## Build error-conditioned decoy pools

In [ ]:
pool_score = build_error_conditioned_pools(
    train_scores_raw, train_labels_raw, NUM_CLASSES, verbose=True)


## Strategy comparison — configuration & helper functions

In [ ]:
NOISE_STD = 0.5

STRATEGIES = [
    ('score_coord',            0.0),
    ('score_coord_noise',      NOISE_STD),
    ('nearest_neighbor',       0.0),
    ('nearest_neighbor_noise', NOISE_STD),
    ('full_vector',            0.0),
    ('full_vector_noise',      NOISE_STD),
    ('binary_coord',           0.0),
    ('binary_coord_noise',     NOISE_STD),
]
STRATEGY_LABELS = {
    'score_coord':            'Random (SC)',
    'score_coord_noise':      'Random + noise',
    'nearest_neighbor':       'Nearest-neighbor',
    'nearest_neighbor_noise': 'NN + noise',
    'full_vector':            'Full vector',
    'full_vector_noise':      'Full vector + noise',
    'binary_coord':           'Binary pool',
    'binary_coord_noise':     'Binary pool + noise',
}
STRATEGY_COLORS = {
    'score_coord':            '#1976D2',
    'score_coord_noise':      '#42A5F5',
    'nearest_neighbor':       '#E65100',
    'nearest_neighbor_noise': '#FF8A65',
    'full_vector':            '#2E7D32',
    'full_vector_noise':      '#66BB6A',
    'binary_coord':           '#9C27B0',
    'binary_coord_noise':     '#CE93D8',
}

MAX_NN_POOL = 5000

def build_error_vector_pool(train_scores, train_labels, num_classes,
                            min_pool=30, max_pool_size=MAX_NN_POOL):
    """Pool of full logit vectors for NN and full_vector strategies, capped per class."""
    rng  = np.random.default_rng(0)
    pred = train_scores.argmax(axis=1)
    pool = {}
    for k in range(num_classes):
        err_mask = (pred == k) & (train_labels != k)
        vecs = train_scores[err_mask] if err_mask.sum() >= min_pool \
               else train_scores[train_labels != k]
        if len(vecs) > max_pool_size:
            vecs = vecs[rng.choice(len(vecs), size=max_pool_size, replace=False)]
        pool[k] = vecs
    print(f"  NN vector pool built: {num_classes} classes, "
          f"sizes {min(len(v) for v in pool.values())}-{max(len(v) for v in pool.values())}")
    return pool


def build_binary_pools(train_scores, train_labels, num_classes, max_pool_size=10_000):
    """Binary null pool: pool[c] = score_at_c for ALL train samples with label != c.
    This is the correct null for 'is this sample class c?'"""
    rng = np.random.default_rng(0)
    pool = {}
    for c in range(num_classes):
        neg_mask = train_labels != c
        vals = train_scores[neg_mask, c]
        if len(vals) > max_pool_size:
            vals = vals[rng.choice(len(vals), size=max_pool_size, replace=False)]
        pool[c] = vals
    sizes = [len(v) for v in pool.values()]
    print(f"  Binary pool built: {num_classes} classes, "
          f"sizes {min(sizes)}-{max(sizes)}, median={int(np.median(sizes))}")
    return pool


NN_BATCH = 1024

def _build_decoy_nearest_neighbor(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """NN decoy with batched distance computation to avoid OOM."""
    n, C = sc_np.shape; pred = sc_np.argmax(1); dc = sc_np.copy(); coords = np.arange(C)
    for k in range(C):
        mask = pred == k
        if not mask.any(): continue
        pk = pool_error_vectors.get(k)
        if pk is None or len(pk) == 0: continue
        comp = coords[coords != k]
        S = sc_np[mask][:, comp]; V = pk[:, comp]; V_sq = (V ** 2).sum(1)
        best_idx = np.empty(S.shape[0], dtype=np.intp)
        for start in range(0, S.shape[0], NN_BATCH):
            end = min(start + NN_BATCH, S.shape[0])
            Sb = S[start:end]
            dists = (Sb ** 2).sum(1, keepdims=True) + V_sq[None, :] - 2 * Sb @ V.T
            best_idx[start:end] = np.maximum(dists, 0).argmin(1)
        dc[mask, k] = pk[best_idx, k]
    if noise_std > 0: dc += rng.normal(0, noise_std, dc.shape)
    return dc


def _build_decoy_full_vector(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """Full vector replacement: decoy = random full error vector from pool[c_hat]."""
    pred = sc_np.argmax(axis=1); dc = np.zeros_like(sc_np)
    for k in range(sc_np.shape[1]):
        mask = pred == k
        if not mask.any(): continue
        pk = pool_error_vectors.get(k)
        if pk is None or len(pk) == 0: dc[mask] = sc_np[mask]; continue
        dc[mask] = pk[rng.choice(len(pk), size=mask.sum(), replace=True)]
    if noise_std > 0: dc += rng.normal(0, noise_std, dc.shape)
    return dc


def apply_strategy(scores_np, pool_score, pool_error_vectors, strategy, noise_std,
                   binary_pool=None):
    rng = np.random.default_rng(42)
    _n  = noise_std if strategy.endswith('_noise') else 0.0
    if strategy in ('nearest_neighbor', 'nearest_neighbor_noise'):
        return _build_decoy_nearest_neighbor(scores_np, pool_error_vectors, rng, _n)
    if strategy in ('full_vector', 'full_vector_noise'):
        return _build_decoy_full_vector(scores_np, pool_error_vectors, rng, _n)
    if strategy in ('binary_coord', 'binary_coord_noise'):
        dc = _build_decoy_score_coord(scores_np, binary_pool, rng)
        if _n > 0: dc += rng.normal(0, _n, dc.shape)
        return dc
    dc = _build_decoy_score_coord(scores_np, pool_score, rng)
    if _n > 0: dc += rng.normal(0, _n, dc.shape)
    return dc


def compute_fdr_acc_curves(scores_np, decoy_np, labels_np, pi0=0.0):
    n = len(labels_np)
    pred_sc = scores_np.max(1); pred_lb = scores_np.argmax(1); dc_sc = decoy_np.max(1)
    correct = (pred_lb == labels_np).astype(int)
    sidx = np.argsort(pred_sc); ps = pred_sc[sidx]; cs = correct[sidx]
    FD = 1 - cs; FC = np.cumsum(FD[::-1])[::-1]; DC = np.arange(n, 0, -1)
    QVAL_true = np.clip(np.minimum.accumulate(np.clip(FC / DC, 0, 1)), 0, 1)
    tsc = np.maximum(pred_sc, dc_sc); twin = (pred_sc > dc_sc).astype(int)
    ti = np.argsort(tsc); FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
    DC_t = np.maximum(DC - FC_t, 1)
    QVAL_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)
    sd = np.sort(dc_sc); uz, cz = np.unique(dc_sc, return_counts=True); nuz = len(uz)
    PW = np.clip((np.searchsorted(ps, uz, 'left') - pi0 * np.searchsorted(sd, uz, 'left')) / ((1-pi0)*n), 0, 1)
    PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
    Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
    fdr = np.zeros(n)
    for i, T in enumerate(ps[::-1]):
        D = i + 1; F0 = pi0 * (dc_sc > T).sum()
        zi = np.searchsorted(uz, T, 'left')
        F1 = 0.0 if zi >= nuz else (1-pi0) * (Rj[zi:] * cz[zi:]).sum()
        fdr[i] = (F0 + F1) / D if D > 0 else 0
    QVAL_mm = np.clip(np.minimum.accumulate(np.clip(fdr, 0, 1)[::-1]), 0, 1)
    pi0_tdc = float(np.clip(QVAL_TDC[0], 0, 1))
    pi0_mm  = float(np.clip(QVAL_mm[0],  0, 1))
    At = np.zeros(n); Ae = np.zeros(n); Am = np.zeros(n)
    for i in range(n):
        At[i] = (cs[i:].sum() + (1 - cs[:i]).sum()) / n
        acc = n - i
        Ae[i] = np.clip((acc*(1 - QVAL_TDC[i]) + n*pi0_tdc - acc*QVAL_TDC[i]) / n, 0, 1)
        Am[i] = np.clip((acc*(1 - QVAL_mm[i])  + n*pi0_mm  - acc*QVAL_mm[i])  / n, 0, 1)
    At = np.clip(At, 0, 1)
    r  = np.arange(n) / n
    tp = int(cs.sum()); TPi = np.cumsum(cs[::-1])[::-1]; Di = DC
    acc_st = float(At[0]); acc_ta = float(At.max())
    acc_st_mm = float(Am[0]); acc_ta_mm = float(Am.max())
    return dict(
        normalized_rank=r, pred_scores_sorted=ps, pred_scores=pred_sc, decoy_scores=dc_sc,
        QVAL_true=QVAL_true, QVAL_TDC=QVAL_TDC, QVAL_mixmax=QVAL_mm,
        Acc_true=At, Acc_est=Ae, Acc_est_MM=Am,
        precision_true=np.where(Di>0, TPi/Di, 0), recall_true=TPi/max(tp,1),
        precision_est=np.clip(1-QVAL_mm, 0, 1),
        recall_est=np.clip((1-QVAL_mm)*Di/max(tp,1), 0, 1),
        correct=correct, pred_label=pred_lb, labels=labels_np,
        true_acc=float(correct.mean()), acc_st_true=acc_st, acc_ta_true=acc_ta,
        acc_st_est_mm=acc_st_mm, acc_ta_est_mm=acc_ta_mm,
        err_st_mm=abs(acc_st_mm - acc_st), err_ta_mm=abs(acc_ta_mm - acc_ta), n=n,
    )

## Build NN pool and prepare test arrays

In [ ]:
print("Building NN error vector pool...")
pool_error_vectors = build_error_vector_pool(train_scores_raw, train_labels_raw, NUM_CLASSES)

print("Building binary pools (label != c for each c)...")
binary_pool_score = build_binary_pools(train_scores_raw, train_labels_raw, NUM_CLASSES)

sc_np = test_logits.numpy()
ft_np = test_features.numpy()
lb_np = test_labels_t.numpy()

print(f"Test: n={len(lb_np)}  acc={(sc_np.argmax(1)==lb_np).mean():.4f}")
n_strats = len(STRATEGIES)

## Run raw decoy strategies

In [ ]:
raw_results = {}
for strat_name, noise_std in STRATEGIES:
    dc   = apply_strategy(sc_np, pool_score, pool_error_vectors, strat_name, noise_std,
                          binary_pool=binary_pool_score)
    crv  = compute_fdr_acc_curves(sc_np, dc, lb_np)
    raw_results[strat_name] = crv
    print(f"  {strat_name:<25}  true_acc={crv['true_acc']:.3f}  "
          f"err_st={crv['err_st_mm']:.3f}  err_ta={crv['err_ta_mm']:.3f}")

### Class-wise FDR diagnostic

Evaluate FDR/Acc **per predicted class** for the top 3 most abundant classes.  
- If per-class curves are good but aggregate is bad → problem is cross-class competition (many classes dilute decoy quality).  
- If per-class curves are already bad → pool itself doesn't match test distribution for that class.

In [ ]:
# Find top 3 most-predicted classes
pred_test = sc_np.argmax(1)
class_counts = np.bincount(pred_test, minlength=NUM_CLASSES)
top3 = np.argsort(class_counts)[::-1][:3]
print("Top 3 predicted classes:")
for c in top3:
    n_c = class_counts[c]
    acc_c = (pred_test[pred_test == c] == lb_np[pred_test == c]).mean()
    print(f"  class {c}: n={n_c}  ({n_c/len(pred_test)*100:.1f}%)  acc={acc_c:.3f}  pool_score n={len(pool_score[c])}")

# Compute per-class FDR for score_coord strategy (representative)
strat_for_classwise = 'score_coord'
dc_full = apply_strategy(sc_np, pool_score, pool_error_vectors, strat_for_classwise, 0.0,
                          binary_pool=binary_pool_score)

classwise = {}
for c in top3:
    mask = pred_test == c
    if mask.sum() < 20:
        print(f"  class {c}: too few samples ({mask.sum()}), skipping")
        continue
    classwise[c] = compute_fdr_acc_curves(sc_np[mask], dc_full[mask], lb_np[mask])

# Also compute aggregate for comparison
agg = raw_results[strat_for_classwise]

# ── Plot: per-class FDR + Acc vs aggregate ───────────────────────────────────
n_cls = len(classwise)
fig, axes = plt.subplots(2, n_cls + 1, figsize=(5 * (n_cls + 1), 8))

colors_cls = ['#E53935', '#1E88E5', '#43A047']

# Aggregate column
ax = axes[0][0]
r = agg['normalized_rank']
ax.plot(r, agg['QVAL_true'],   'k--', lw=2,   label='True FDR')
ax.plot(r, agg['QVAL_mixmax'], color='#1976D2', lw=1.8, label=f'MixMax err={agg["err_st_mm"]:.3f}')
ax.set_title(f'ALL CLASSES (n={agg["n"]})\nacc={agg["true_acc"]:.3f}')
ax.set_ylabel('q-value'); ax.set_xlabel('Fraction accepted')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)

ax = axes[1][0]
ax.plot(r, agg['Acc_true'],   'k--', lw=2,   label='True Acc')
ax.plot(r, agg['Acc_est_MM'], color='#1976D2', lw=1.8, label=f'MixMax err={agg["err_ta_mm"]:.3f}')
ax.set_ylabel('Accuracy'); ax.set_xlabel('Fraction accepted')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# Per-class columns
for col_i, (c, color) in enumerate(zip(classwise.keys(), colors_cls)):
    cv = classwise[c]
    r_c = cv['normalized_rank']
    n_c = cv['n']; acc_c = cv['true_acc']

    ax = axes[0][col_i + 1]
    ax.plot(r_c, cv['QVAL_true'],   'k--', lw=2,  label='True FDR')
    ax.plot(r_c, cv['QVAL_mixmax'], color=color, lw=1.8, label=f'MixMax err={cv["err_st_mm"]:.3f}')
    ax.set_title(f'CLASS {c} (n={n_c})\nacc={acc_c:.3f}  pool={len(pool_score[c])}')
    ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)

    ax = axes[1][col_i + 1]
    ax.plot(r_c, cv['Acc_true'],   'k--', lw=2,  label='True Acc')
    ax.plot(r_c, cv['Acc_est_MM'], color=color, lw=1.8, label=f'MixMax err={cv["err_ta_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

plt.suptitle(f'Plant — Class-wise FDR/Acc  (strategy: {strat_for_classwise})', fontsize=14)
plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'Class':<12} {'n':>6}  {'acc':>6}  {'pool_n':>7}  {'MAE_ST':>8}  {'MAE_TA':>8}")
print('-' * 55)
print(f"{'ALL':<12} {agg['n']:>6}  {agg['true_acc']:>6.3f}  {'':>7}  {agg['err_st_mm']:>8.4f}  {agg['err_ta_mm']:>8.4f}")
for c in classwise:
    cv = classwise[c]
    print(f"{'class '+str(c):<12} {cv['n']:>6}  {cv['true_acc']:>6.3f}  {len(pool_score[c]):>7}  {cv['err_st_mm']:>8.4f}  {cv['err_ta_mm']:>8.4f}")

### Per-class scatter: model score vs decoy score at coord c

For each of the top 3 classes, scatter `score[i, c]` (model logit at the predicted class) vs `decoy[i, c]` (the replaced coord).  
This shows directly whether the pool draws land around the diagonal for incorrect predictions — the core exchangeability requirement.

In [ ]:
# Per-class scatter: model score_c vs decoy score_c (at the predicted class coord)
dc_sc = dc_full  # score_coord decoys already computed above

fig, axes = plt.subplots(1, len(top3), figsize=(6 * len(top3), 5))
if len(top3) == 1: axes = [axes]

for ax, c in zip(axes, top3):
    mask = pred_test == c
    correct_c = (pred_test[mask] == lb_np[mask])

    model_at_c = sc_np[mask, c]
    decoy_at_c = dc_sc[mask, c]

    # Pool distribution for reference
    pool_c = pool_score[c]

    ax.scatter(model_at_c[correct_c],  decoy_at_c[correct_c],
               s=8, alpha=0.3, color='steelblue', label=f'correct ({correct_c.sum()})', rasterized=True)
    ax.scatter(model_at_c[~correct_c], decoy_at_c[~correct_c],
               s=12, alpha=0.6, color='crimson', label=f'incorrect ({(~correct_c).sum()})', rasterized=True)

    lo = min(model_at_c.min(), decoy_at_c.min(), pool_c.min()) - 0.3
    hi = max(model_at_c.max(), decoy_at_c.max(), pool_c.max()) + 0.3
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)

    # Show pool range as shaded band on y-axis
    p5, p95 = np.percentile(pool_c, [5, 95])
    ax.axhspan(p5, p95, alpha=0.12, color='orange', label=f'pool 5-95% [{p5:.1f}, {p95:.1f}]')
    ax.axhline(np.median(pool_c), color='orange', ls=':', lw=1.5, alpha=0.7, label=f'pool median={np.median(pool_c):.1f}')

    # Annotate median shift
    med_model_inc = np.median(model_at_c[~correct_c]) if (~correct_c).sum() > 0 else np.nan
    med_pool = np.median(pool_c)
    ax.set_title(f'Class {c}  (n={mask.sum()}, acc={correct_c.mean():.2f})\n'
                 f'pool_med={med_pool:.2f}  incorr_model_med={med_model_inc:.2f}  '
                 f'shift={med_pool - med_model_inc:+.2f}')
    ax.set_xlabel(f'Model score at class {c}')
    ax.set_ylabel(f'Decoy score at class {c}')
    ax.legend(fontsize=8, loc='upper left')
    ax.set_aspect('equal', 'box')
    ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Per-class scatter: model vs decoy at coord c  (score_coord strategy)', fontsize=13)
plt.tight_layout(); plt.show()

### Margin diagnostic: top-1 vs top-2 logit gap

`score_coord` only replaces one coordinate. The decoy's max logit is:  
`max(decoy) = max(second_highest_model_logit, pool_draw_at_c)`

If the **margin** (top1 − top2) is small, the pool draw almost never matters — `max(decoy) ≈ second_highest` regardless.  
This makes the decoy score deterministic (ignoring the pool) and breaks exchangeability.

On ~10 classes (BREEDS) margins are large. On ~500 classes (plant) margins can be tiny — the model is uncertain between many similar classes.

In [ ]:
# ── Margin analysis ───────────────────────────────────────────────────────────
sorted_logits = np.sort(sc_np, axis=1)[:, ::-1]  # descending per sample
top1    = sorted_logits[:, 0]
top2    = sorted_logits[:, 1]
margin  = top1 - top2

correct_mask = (sc_np.argmax(1) == lb_np)

# Also compute: what fraction of the time does the pool draw even matter?
# pool_draw matters when pool_draw > second_highest (i.e. it becomes max(decoy))
# otherwise max(decoy) = second_highest regardless of pool
dc_sc_full = dc_full  # score_coord decoys
decoy_at_c = np.array([dc_sc_full[i, sc_np[i].argmax()] for i in range(len(sc_np))])
pool_wins_over_top2 = decoy_at_c > top2  # pool draw became the max of decoy vector

print(f"=== MARGIN ANALYSIS (test set, n={len(sc_np)}) ===\n")
print(f"  Margin (top1 - top2):")
print(f"    all:       mean={margin.mean():.3f}  median={np.median(margin):.3f}  "
      f"std={margin.std():.3f}  <0.5: {(margin < 0.5).mean()*100:.1f}%  <1.0: {(margin < 1.0).mean()*100:.1f}%")
print(f"    correct:   mean={margin[correct_mask].mean():.3f}  median={np.median(margin[correct_mask]):.3f}")
print(f"    incorrect: mean={margin[~correct_mask].mean():.3f}  median={np.median(margin[~correct_mask]):.3f}")
print(f"\n  Pool draw relevance (pool_draw_at_c > second_highest → pool controls max(decoy)):")
print(f"    all:       {pool_wins_over_top2.mean()*100:.1f}%")
print(f"    correct:   {pool_wins_over_top2[correct_mask].mean()*100:.1f}%")
print(f"    incorrect: {pool_wins_over_top2[~correct_mask].mean()*100:.1f}%")
print(f"\n  → When pool draw < top2, max(decoy) = top2 regardless of pool quality.")
print(f"    This means for {(~pool_wins_over_top2).mean()*100:.1f}% of samples, "
      f"the pool draw is invisible to FDR.")

# ── Plot 1: Margin histogram ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

ax = axes[0]
bins_m = np.linspace(0, max(10, np.percentile(margin, 99)), 60)
ax.hist(margin[correct_mask],  bins=bins_m, density=True, alpha=0.6, color='steelblue',
        label=f'correct (med={np.median(margin[correct_mask]):.2f})')
ax.hist(margin[~correct_mask], bins=bins_m, density=True, alpha=0.6, color='crimson',
        label=f'incorrect (med={np.median(margin[~correct_mask]):.2f})')
ax.axvline(0.5, color='black', ls=':', lw=1.2, label='margin=0.5')
ax.axvline(1.0, color='gray',  ls=':', lw=1.0, label='margin=1.0')
ax.set_xlabel('Margin (top1 − top2)'); ax.set_ylabel('Density')
ax.set_title(f'Margin distribution\n<0.5: {(margin<0.5).mean()*100:.0f}%  '
             f'<1.0: {(margin<1.0).mean()*100:.0f}%')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# ── Plot 2: max(model) vs max(decoy) colored by whether pool matters ─────────
ax = axes[1]
max_model = sc_np.max(1)
max_decoy = dc_sc_full.max(1)
idx = np.random.default_rng(0).choice(len(max_model), min(4000, len(max_model)), replace=False)

pw = pool_wins_over_top2[idx]
ax.scatter(max_model[idx][~pw], max_decoy[idx][~pw],
           s=5, alpha=0.3, color='gray', label=f'pool invisible ({(~pw).sum()})', rasterized=True)
ax.scatter(max_model[idx][pw],  max_decoy[idx][pw],
           s=5, alpha=0.5, color='crimson', label=f'pool controls max(decoy) ({pw.sum()})', rasterized=True)
lims = [min(max_model.min(), max_decoy.min()) - 0.2, max(max_model.max(), max_decoy.max()) + 0.2]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel('max(model logits)'); ax.set_ylabel('max(decoy logits)')
ax.set_title(f'max(model) vs max(decoy)\npool invisible: {(~pool_wins_over_top2).mean()*100:.0f}%')
ax.legend(fontsize=8, markerscale=3); ax.grid(ls='--', alpha=0.3)

# ── Plot 3: Margin vs pool_draw - top2 (how much does pool "overshoot"?) ─────
ax = axes[2]
pool_overshoot = decoy_at_c - top2
ax.scatter(margin[idx], pool_overshoot[idx],
           s=5, alpha=0.3, color='steelblue', rasterized=True)
ax.axhline(0, color='black', ls='--', lw=1, alpha=0.6)
ax.axvline(0, color='black', ls='--', lw=1, alpha=0.6)
ax.set_xlabel('Margin (top1 − top2)')
ax.set_ylabel('Pool draw − top2')
ax.set_title('Pool overshoot vs margin\n(above 0 → pool controls max(decoy))')
ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Margin & pool-draw relevance diagnostic', fontsize=13)
plt.tight_layout(); plt.show()

# ── Per-class margin for top 3 ────────────────────────────────────────────────
print(f"\nPer-class margin for top-3 predicted classes:")
print(f"  {'class':<10} {'n':>6}  {'margin_med':>10}  {'margin<0.5':>10}  {'pool_relevant':>14}")
print(f"  {'-'*56}")
for c in top3:
    m = pred_test == c
    mg = margin[m]
    pr = pool_wins_over_top2[m]
    print(f"  class {c:<4d} {m.sum():>6}  {np.median(mg):>10.3f}  "
          f"{(mg < 0.5).mean()*100:>9.1f}%  {pr.mean()*100:>13.1f}%")

### Binary FDR for class 204 — proper binary pool

Two pools compared:
- **Binary pool** (green): `train_scores[label ≠ 204, 204]` — logit at class 204 for ALL non-204 training samples. This is the correct null for "is this sample class 204?"
- **Old pool** (orange): `train_scores[argmax=204 AND label≠204, 204]` — only confident errors. Too narrow for binary testing.

Under H₀ (sample is not 204), the binary pool p-values should be ~Uniform[0,1].

In [ ]:
TARGET_CLASS = 204

# ── Scores and labels (binary) ────────────────────────────────────────────────
model_score_c = sc_np[:, TARGET_CLASS]
is_positive   = (lb_np == TARGET_CLASS)
n_all = len(model_score_c)
n_pos = is_positive.sum()
n_neg = n_all - n_pos
print(f"Binary task: class {TARGET_CLASS}")
print(f"  n={n_all}  positives={n_pos}  negatives={n_neg}")

# ── Binary pool: score_204 for ALL train samples with label != 204 ────────────
binary_pool = train_scores_raw[train_labels_raw != TARGET_CLASS, TARGET_CLASS]
MAX_BINARY_POOL = 20_000
if len(binary_pool) > MAX_BINARY_POOL:
    binary_pool = np.random.default_rng(0).choice(binary_pool, MAX_BINARY_POOL, replace=False)
print(f"  Binary pool: n={len(binary_pool)}  [{binary_pool.min():.2f}, {binary_pool.max():.2f}]  "
      f"median={np.median(binary_pool):.2f}")

old_pool = pool_score[TARGET_CLASS]
print(f"  Old pool:    n={len(old_pool)}  [{old_pool.min():.2f}, {old_pool.max():.2f}]  "
      f"median={np.median(old_pool):.2f}")

# ── Decoy draws ───────────────────────────────────────────────────────────────
rng_bin = np.random.default_rng(42)
decoy_binary = rng_bin.choice(binary_pool, size=n_all, replace=True)
decoy_old    = rng_bin.choice(old_pool,    size=n_all, replace=True)

# ── p-values ──────────────────────────────────────────────────────────────────
def pvals_from_pool(scores, pool):
    ps = np.sort(pool)
    pv = 1.0 - np.searchsorted(ps, scores, side='right') / len(ps)
    return np.clip(pv, 1.0 / len(ps), 1.0)

pv_binary = pvals_from_pool(model_score_c, binary_pool)
pv_old    = pvals_from_pool(model_score_c, old_pool)

# ── BH q-values ──────────────────────────────────────────────────────────────
def bh_qvalues(pvals):
    n = len(pvals)
    si = np.argsort(pvals)
    qv = np.zeros(n)
    qv_sorted = np.minimum.accumulate((pvals[si] * n / np.arange(1, n + 1))[::-1])[::-1]
    qv[si] = np.clip(qv_sorted, 0, 1)
    return qv

qv_bh_binary = bh_qvalues(pv_binary)
qv_bh_old    = bh_qvalues(pv_old)

# ── Mix-Max + TDC (1D) ───────────────────────────────────────────────────────
sidx = np.argsort(model_score_c)
ms_sorted = model_score_c[sidx]
cs_sorted = is_positive[sidx].astype(int)
n = n_all
DC_all = np.arange(n, 0, -1)

FD_true = np.cumsum((1 - cs_sorted)[::-1])[::-1]
QVAL_true = np.clip(np.minimum.accumulate(np.clip(FD_true / DC_all, 0, 1)), 0, 1)

def compute_mm_tdc_1d(model_scores_sorted, decoy_draws):
    n = len(model_scores_sorted)
    DC = np.arange(n, 0, -1)
    ms_full = model_score_c; ds_full = decoy_draws
    tsc = np.maximum(ms_full, ds_full)
    twin = (ms_full > ds_full).astype(int)
    ti = np.argsort(tsc)
    FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
    DC_t = np.maximum(DC - FC_t, 1)
    Q_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)
    sd = np.sort(ds_full); uz, cz = np.unique(ds_full, return_counts=True); nuz = len(uz)
    PW = np.clip(np.searchsorted(model_scores_sorted, uz, 'left') / n, 0, 1)
    PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
    Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
    fdr_mm = np.zeros(n)
    for i, T in enumerate(model_scores_sorted[::-1]):
        D = i + 1; zi = np.searchsorted(uz, T, 'left')
        fdr_mm[i] = (0.0 if zi >= nuz else (Rj[zi:] * cz[zi:]).sum()) / D if D > 0 else 0
    Q_MM = np.clip(np.minimum.accumulate(np.clip(fdr_mm, 0, 1)[::-1]), 0, 1)
    return Q_MM, Q_TDC

QVAL_mm_bin, QVAL_tdc_bin = compute_mm_tdc_1d(ms_sorted, decoy_binary)
QVAL_mm_old, QVAL_tdc_old = compute_mm_tdc_1d(ms_sorted, decoy_old)
r = np.arange(n) / n

# ── PLOTS ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Scatter with binary pool
ax = axes[0][0]
idx = np.random.default_rng(0).choice(n_all, min(5000, n_all), replace=False)
pos_i = idx[is_positive[idx]]; neg_i = idx[~is_positive[idx]]
ax.scatter(model_score_c[neg_i], decoy_binary[neg_i],
           s=3, alpha=0.15, color='gray', label=f'neg ({len(neg_i)})', rasterized=True)
ax.scatter(model_score_c[pos_i], decoy_binary[pos_i],
           s=10, alpha=0.6, color='crimson', label=f'pos ({len(pos_i)})', rasterized=True)
lo = min(model_score_c[idx].min(), decoy_binary[idx].min()) - 0.5
hi = max(model_score_c[idx].max(), decoy_binary[idx].max()) + 0.5
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel(f'Model logit at {TARGET_CLASS}'); ax.set_ylabel('Decoy (binary pool)')
ax.set_title(f'Binary pool scatter\npool=train logits at {TARGET_CLASS} where label≠{TARGET_CLASS}')
ax.legend(fontsize=9, markerscale=2); ax.grid(ls='--', alpha=0.3)

# (0,1) Score distributions
ax = axes[0][1]
lo_b = min(model_score_c.min(), binary_pool.min(), old_pool.min()) - 1
hi_b = max(model_score_c.max(), binary_pool.max(), old_pool.max()) + 1
bins_s = np.linspace(lo_b, hi_b, 80)
ax.hist(model_score_c[is_positive],  bins=bins_s, density=True, alpha=0.5, color='crimson',
        label=f'pos model (n={n_pos})')
ax.hist(model_score_c[~is_positive], bins=bins_s, density=True, alpha=0.2, color='gray',
        label=f'neg model (n={n_neg})')
ax.hist(binary_pool, bins=bins_s, density=True, histtype='step', lw=2.5, color='#2E7D32',
        label=f'BINARY pool (n={len(binary_pool)})')
ax.hist(old_pool,    bins=bins_s, density=True, histtype='step', lw=2, color='orange', ls='--',
        label=f'Old pool (n={len(old_pool)})')
ax.set_xlabel(f'Logit at {TARGET_CLASS}'); ax.set_ylabel('Density')
ax.set_title(f'Distributions\nGreen=binary pool  Orange=old error-cond pool')
ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (0,2) p-value histograms
ax = axes[0][2]
bins_p = np.linspace(0, 1, 31)
ax.hist(pv_binary[~is_positive], bins=bins_p, density=True, alpha=0.5, color='#2E7D32',
        label=f'Neg binary (mean={pv_binary[~is_positive].mean():.2f})')
ax.hist(pv_old[~is_positive],    bins=bins_p, density=True, alpha=0.3, color='orange',
        label=f'Neg old (mean={pv_old[~is_positive].mean():.2f})')
ax.hist(pv_binary[is_positive],  bins=bins_p, density=True, alpha=0.5, color='crimson',
        label=f'Pos binary (mean={pv_binary[is_positive].mean():.2f})')
ax.axhline(1, ls='--', color='black', lw=1, label='Uniform')
ax.set_xlabel('p-value'); ax.set_ylabel('Density')
ax.set_title('p-values: binary (green) vs old (orange)\nneg should be ~Uniform')
ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (1,0) FDR curves
ax = axes[1][0]
ax.plot(r, QVAL_true,    'k--', lw=2.5, label='True FDR')
ax.plot(r, QVAL_mm_bin,  color='#2E7D32', lw=2,   label='MixMax (binary)')
ax.plot(r, QVAL_mm_old,  color='orange',  lw=1.5, ls='-.', label='MixMax (old)')
bh_bin_sorted = qv_bh_binary[sidx]
bh_old_sorted = qv_bh_old[sidx]
ax.plot(r, bh_bin_sorted, color='#2E7D32', lw=1.2, ls=':', label='BH (binary)')
ax.plot(r, bh_old_sorted, color='orange',  lw=1.2, ls=':', label='BH (old)')
ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value')
ax.set_ylim(0, 1.05)
ax.set_title(f'FDR: binary pool vs old pool\nclass {TARGET_CLASS}')
ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (1,1) Zoom on right tail
ax = axes[1][1]
zoom = r > 0.9
ax.plot(r[zoom], QVAL_true[zoom],       'k--', lw=2.5, label='True FDR')
ax.plot(r[zoom], QVAL_mm_bin[zoom],     color='#2E7D32', lw=2, label='MixMax (binary)')
ax.plot(r[zoom], QVAL_mm_old[zoom],     color='orange', lw=1.5, ls='-.', label='MixMax (old)')
ax.plot(r[zoom], bh_bin_sorted[zoom],   color='#2E7D32', lw=1.2, ls=':', label='BH (binary)')
ax.set_ylim(0, 1.05)
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value')
ax.set_title('ZOOM: top 10% (where positives live)')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,2) Precision-Recall
ax = axes[1][2]
tp_cum = np.cumsum(cs_sorted[::-1])[::-1]
prec_true = tp_cum / DC_all; rec_true = tp_cum / max(n_pos, 1)
prec_mm = np.clip(1 - QVAL_mm_bin, 0, 1); rec_mm = prec_mm * DC_all / max(n_pos, 1)
prec_bh = np.clip(1 - bh_bin_sorted, 0, 1); rec_bh = prec_bh * DC_all / max(n_pos, 1)
ax.plot(rec_true, prec_true, 'k--', lw=2.5, label='True PR')
ax.plot(rec_mm, prec_mm, color='#2E7D32', lw=2, label='MixMax (binary)')
ax.plot(rec_bh, prec_bh, color='#2E7D32', lw=1.2, ls=':', label='BH (binary)')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.set_title(f'Precision-Recall — class {TARGET_CLASS}')
ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle(f'Binary FDR — class {TARGET_CLASS}  (n={n_all}, pos={n_pos})\n'
             f'Green=binary pool (label≠{TARGET_CLASS})  Orange=old error-conditioned pool', fontsize=13)
plt.tight_layout(); plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\nDiscoveries at controlled FDR:")
print(f"  {'alpha':<7} {'BH(bin)':>12} {'BH(old)':>12} {'MM(bin)':>12} {'MM(old)':>12}   (/{n_pos} pos)")
print(f"  {'-'*62}")
for alpha in [0.01, 0.05, 0.10, 0.20]:
    bb = (qv_bh_binary <= alpha); bb_d = bb.sum(); bb_tp = bb[is_positive].sum()
    bo = (qv_bh_old <= alpha);    bo_d = bo.sum(); bo_tp = bo[is_positive].sum()
    mb = QVAL_mm_bin <= alpha;    mb_d = mb.sum(); mb_tp = cs_sorted[mb].sum()
    mo = QVAL_mm_old <= alpha;    mo_d = mo.sum(); mo_tp = cs_sorted[mo].sum()
    print(f"  {alpha:<7.2f} {bb_d:>5}({bb_tp:>3}tp) {bo_d:>5}({bo_tp:>3}tp) "
          f"{mb_d:>5}({mb_tp:>3}tp) {mo_d:>5}({mo_tp:>3}tp)")

### Figure 1 — Score distributions

In [ ]:
ref = raw_results['score_coord']
bins = np.linspace(ref['pred_scores'].min() - 0.3, ref['pred_scores'].max() + 0.3, 60)
inc_mask = ref['correct'] == 0

ncols = 4; nrows = (n_strats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes_flat = axes.flatten()
for i, (strat_name, _) in enumerate(STRATEGIES):
    ax = axes_flat[i]; c = raw_results[strat_name]; color = STRATEGY_COLORS[strat_name]
    sns.histplot(c['pred_scores'],  bins=bins, stat='density', color='steelblue',
                 kde=True, fill=True, alpha=0.3, label='model', ax=ax)
    sns.histplot(c['decoy_scores'], bins=bins, stat='density', color=color,
                 kde=True, fill=True, alpha=0.4, label='decoy', ax=ax)
    if inc_mask.any():
        sns.histplot(c['pred_scores'][inc_mask], bins=bins, stat='density', color='crimson',
                     kde=True, fill=True, alpha=0.3, label='incorrect', ax=ax)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\ntrue_acc={c["true_acc"]:.3f}', fontsize=10)
    ax.set_xlabel('Max logit'); ax.legend(fontsize=6)
for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
plt.suptitle('Plant — Score Distributions (Raw Decoy)', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 2 — FDR curves

In [ ]:
ncols = 4; nrows = (n_strats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes_flat = axes.flatten()
for i, (strat_name, _) in enumerate(STRATEGIES):
    ax = axes_flat[i]; c = raw_results[strat_name]; r = c['normalized_rank']
    ax.plot(r, c['QVAL_true'],   color='gray',                     lw=1.5, ls='--', label='True FDR')
    ax.plot(r, c['QVAL_mixmax'], color=STRATEGY_COLORS[strat_name], lw=1.8,         label='Mix-Max')
    ax.plot(r, c['QVAL_TDC'],   color='navy',                     lw=1.2, ls=':',  label='TDC')
    ax.axhline(0.1, color='black', lw=0.8, ls=':', alpha=0.4)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nerr_st={c["err_st_mm"]:.3f}', fontsize=10)
    ax.set_xlabel('Fraction accepted'); ax.legend(fontsize=6)
    ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
plt.suptitle('Plant — FDR Curves', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 3 — Accuracy curves overlay

In [ ]:
r = ref['normalized_rank']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(r, ref['Acc_true'], color='black', lw=2, ls='--', label='True Acc')
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    ax.plot(r, c['Acc_est_MM'], color=STRATEGY_COLORS[strat_name], lw=1.5,
            label=f'{STRATEGY_LABELS[strat_name]} ({c["err_st_mm"]:.3f})')
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy curves (MixMax)'); ax.legend(fontsize=6, ncol=2); ax.grid(ls='--', alpha=0.4)

ax = axes[1]
ax.plot(r, ref['QVAL_true'], color='black', lw=2, ls='--', label='True FDR')
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    ax.plot(r, c['QVAL_mixmax'], color=STRATEGY_COLORS[strat_name], lw=1.5, label=STRATEGY_LABELS[strat_name])
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value')
ax.set_title('FDR curves comparison'); ax.legend(fontsize=6, ncol=2)
ax.grid(ls='--', alpha=0.4); ax.set_ylim(0, 1.05)

plt.suptitle('Plant — All Strategies Comparison', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 4 — Scatter: decoy vs model score (TEST)

In [ ]:
corr_test = raw_results['score_coord']['correct'].astype(bool)
ncols = 4; nrows = (n_strats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes_flat = axes.flatten()
for i, (strat_name, _) in enumerate(STRATEGIES):
    ax = axes_flat[i]; c = raw_results[strat_name]; ms = c['pred_scores']; ds = c['decoy_scores']
    idx = np.random.default_rng(0).choice(len(ms), min(3000, len(ms)), replace=False)
    cor = corr_test[idx]
    ax.scatter(ms[idx][cor],  ds[idx][cor],  s=6, alpha=0.3, color='steelblue', label='correct',  rasterized=True)
    ax.scatter(ms[idx][~cor], ds[idx][~cor], s=6, alpha=0.5, color='crimson',   label='incorrect', rasterized=True)
    lims = [min(ms.min(), ds.min()) - 0.1, max(ms.max(), ds.max()) + 0.1]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(STRATEGY_LABELS[strat_name], fontsize=10); ax.set_xlabel('Model score')
    ax.legend(fontsize=6); ax.grid(ls='--', alpha=0.3)
for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
plt.suptitle('Plant — Decoy vs Model Score (TEST)', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 5 — Scatter: decoy vs model score (TRAINING)

Diagnoses distribution shift: if training looks OK (dots straddle the diagonal) but test is all below → pool doesn't cover test distribution.

In [ ]:
tr_sc = train_scores_raw; tr_lb = train_labels_raw
corr_tr = tr_sc.argmax(1) == tr_lb
ncols = 4; nrows = (n_strats + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes_flat = axes.flatten()
for i, (strat_name, noise_std) in enumerate(STRATEGIES):
    ax = axes_flat[i]
    dc = apply_strategy(tr_sc, pool_score, pool_error_vectors, strat_name, noise_std,
                        binary_pool=binary_pool_score)
    ms = tr_sc.max(1); ds = dc.max(1)
    frac_above = (ds[~corr_tr] > ms[~corr_tr]).mean()
    idx = np.random.default_rng(0).choice(len(ms), min(3000, len(ms)), replace=False)
    cor = corr_tr[idx]
    ax.scatter(ms[idx][cor],  ds[idx][cor],  s=6, alpha=0.3, color='steelblue', label='correct',  rasterized=True)
    ax.scatter(ms[idx][~cor], ds[idx][~cor], s=6, alpha=0.6, color='crimson',   label='incorrect', rasterized=True)
    lims = [min(ms.min(), ds.min()) - 0.1, max(ms.max(), ds.max()) + 0.1]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5); ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\ndecoy>model(incorr):{frac_above:.2f}', fontsize=10)
    ax.set_xlabel('Model score'); ax.legend(fontsize=6); ax.grid(ls='--', alpha=0.3)
for j in range(i+1, len(axes_flat)): axes_flat[j].set_visible(False)
plt.suptitle('Plant — TRAINING scatter (Decoy vs Model)', fontsize=13)
plt.tight_layout(); plt.show()

### Figure 6 — Pool coverage diagnostic

Ideal: `pool ≥ incorrect` ≈ 0.5 (pool looks like the test error distribution). A shift means pool is too low/high relative to test.

In [ ]:
pred_cls_test = sc_np.argmax(1); corr_test_b = pred_cls_test == lb_np
s_chat = sc_np[np.arange(len(sc_np)), pred_cls_test]
tr_pred = tr_sc.argmax(1); tr_err = tr_pred != tr_lb
all_pool = np.concatenate([pool_score[c] for c in range(NUM_CLASSES)])
sc_tr_err = tr_sc[tr_err][np.arange(tr_err.sum()), tr_pred[tr_err]]

frac_pool = np.mean([np.mean(pool_score[c] >= s)
                     for s, c in zip(s_chat[~corr_test_b], pred_cls_test[~corr_test_b])
                     ]) if (~corr_test_b).sum() > 0 else float('nan')
shift = np.median(all_pool) - np.median(s_chat[~corr_test_b])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.hist(sc_tr_err, bins=50, density=True, alpha=0.6, color='crimson',  label=f'train errors at c_hat (n={tr_err.sum()})')
ax.hist(all_pool,  bins=50, density=True, alpha=0.5, color='orange',   label=f'pool values (n={len(all_pool)})')
ax.set_title('TRAIN: error scores vs pool'); ax.set_xlabel('Score at c_hat'); ax.set_ylabel('Density')
ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.hist(s_chat[~corr_test_b], bins=50, density=True, alpha=0.6, color='crimson',   label='test incorrect')
ax.hist(s_chat[corr_test_b],  bins=50, density=True, alpha=0.4, color='steelblue', label='test correct')
ax.hist(all_pool,             bins=50, density=True, alpha=0.4, color='orange',    label='pool', linestyle='--')
ax.set_title(f'TEST: scores vs pool\npool ≥ incorrect: {frac_pool:.2f}  pool-median shift: {shift:+.2f}')
ax.set_xlabel('Score at c_hat'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Pool coverage diagnostic', fontsize=13)
plt.tight_layout(); plt.show()
print(f'Pool coverage:  pool ≥ incorrect test: {frac_pool:.3f}  (ideal 0.5)')
print(f'  pool median: {np.median(all_pool):.3f}  incorrect test median: {np.median(s_chat[~corr_test_b]):.3f}  shift: {shift:+.3f}')


### Figure 7 — P-value diagnostics & pool calibration

Under H₀: `p_i = P(pool[c_hat] ≥ score_at_c_hat)` should be Uniform[0,1] for incorrect predictions.

In [ ]:
def compute_pvalues(sc_np, lb_np, pool_score):
    pred = sc_np.argmax(1); s = sc_np[np.arange(len(sc_np)), pred]
    pv = np.array([np.mean(pool_score[c] >= si) for si, c in zip(s, pred)])
    return pv, pred == lb_np, pred

def build_calibrated_pool(pool_score, test_scores_per_class, num_classes):
    """Shift+scale pool per class to match test median + IQR."""
    cal = {}
    for c in range(num_classes):
        pool = pool_score.get(c, np.array([])); tsc = test_scores_per_class.get(c, np.array([]))
        if len(pool) == 0 or len(tsc) == 0: cal[c] = pool; continue
        pm, tm = np.median(pool), np.median(tsc)
        pi = np.percentile(pool, 75) - np.percentile(pool, 25)
        ti = np.percentile(tsc,  75) - np.percentile(tsc,  25)
        cal[c] = pool + (tm - pm) if pi < 1e-6 or ti < 1e-6 else (pool - pm) * (ti / pi) + tm
    return cal

p_raw, corr_b, p_cls = compute_pvalues(sc_np, lb_np, pool_score)
tsc_per_cls = {c: sc_np[p_cls == c, c] for c in range(NUM_CLASSES) if (p_cls == c).sum() > 0}
pool_cal    = build_calibrated_pool(pool_score, tsc_per_cls, NUM_CLASSES)
p_cal, _, _ = compute_pvalues(sc_np, lb_np, pool_cal)

print(f'Raw pool:   incorrect mean_p={p_raw[~corr_b].mean():.3f}  correct mean_p={p_raw[corr_b].mean():.3f}')
print(f'Calibrated: incorrect mean_p={p_cal[~corr_b].mean():.3f}  correct mean_p={p_cal[corr_b].mean():.3f}  (ideal incorrect≈0.5)')

bins_p = np.linspace(0, 1, 21)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

ax = axes[0]
ax.hist(p_raw[corr_b],  bins=bins_p, density=True, alpha=0.6, color='steelblue', label=f'Correct (n={corr_b.sum()})')
ax.hist(p_raw[~corr_b], bins=bins_p, density=True, alpha=0.6, color='crimson',   label=f'Incorrect (n={(~corr_b).sum()})')
ax.axhline(1, ls='--', color='black', lw=1.2, label='Uniform')
ax.set_title(f'Raw pool p-values\nincorrect mean={p_raw[~corr_b].mean():.3f}')
ax.set_xlabel('p-value'); ax.set_ylabel('Density'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.hist(p_cal[corr_b],  bins=bins_p, density=True, alpha=0.6, color='steelblue', label='Correct')
ax.hist(p_cal[~corr_b], bins=bins_p, density=True, alpha=0.6, color='crimson',   label='Incorrect')
ax.axhline(1, ls='--', color='black', lw=1.2, label='Uniform')
ax.set_title(f'Calibrated pool p-values\nincorrect mean={p_cal[~corr_b].mean():.3f}')
ax.set_xlabel('p-value'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[2]
for pv, lbl, col in [(p_raw[~corr_b], 'Raw (H0)', 'crimson'), (p_cal[~corr_b], 'Cal (H0)', 'darkorange')]:
    if len(pv): ax.plot(np.linspace(0,1,len(pv)), np.sort(pv), color=col, lw=1.8, label=lbl)
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.6,label='Uniform')
ax.set_xlabel('Uniform quantile'); ax.set_ylabel('Empirical p-value quantile')
ax.set_title('QQ plot — incorrect preds\n(H0 ideal = diagonal)'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[3]
cr = [p_raw[(p_cls==c)&~corr_b].mean() if ((p_cls==c)&~corr_b).sum()>0 else np.nan for c in range(NUM_CLASSES)]
cc = [p_cal[(p_cls==c)&~corr_b].mean() if ((p_cls==c)&~corr_b).sum()>0 else np.nan for c in range(NUM_CLASSES)]
x = np.arange(NUM_CLASSES)
ax.bar(x-0.2, cr, 0.4, color='crimson',   alpha=0.7, label='Raw pool')
ax.bar(x+0.2, cc, 0.4, color='darkorange', alpha=0.7, label='Calibrated')
ax.axhline(0.5, ls='--', color='black', lw=1, label='Ideal (0.5)')
ax.set_xlabel('Predicted class c_hat'); ax.set_ylabel('Mean p-value (incorrect only)')
ax.set_title('Per-class pool calibration'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — P-value diagnostics', fontsize=13)
plt.tight_layout(); plt.show()


### Effect of pool calibration on FDR / Accuracy

In [ ]:
dc_cal     = apply_strategy(sc_np, pool_cal, pool_error_vectors, 'score_coord', 0.0,
                     binary_pool=binary_pool_score)
curves_cal = compute_fdr_acc_curves(sc_np, dc_cal, lb_np)
sc_ref = raw_results['score_coord']; r = sc_ref['normalized_rank']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(r, sc_ref['QVAL_true'],       'k--', lw=2,   label='True FDR')
ax.plot(r, sc_ref['QVAL_mixmax'],     color='#1976D2',   lw=1.8, label=f'Raw  err_st={sc_ref["err_st_mm"]:.3f}')
ax.plot(r, curves_cal['QVAL_mixmax'], color='darkorange', lw=1.8, label=f'Cal  err_st={curves_cal["err_st_mm"]:.3f}')
ax.set_title('FDR: raw pool vs calibrated'); ax.set_xlabel('Fraction accepted')
ax.set_ylabel('q-value'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

ax = axes[1]
ax.plot(r, sc_ref['Acc_true'],        'k--', lw=2,   label='True Acc')
ax.plot(r, sc_ref['Acc_est_MM'],      color='#1976D2',   lw=1.8, label=f'Raw  err_ta={sc_ref["err_ta_mm"]:.3f}')
ax.plot(r, curves_cal['Acc_est_MM'],  color='darkorange', lw=1.8, label=f'Cal  err_ta={curves_cal["err_ta_mm"]:.3f}')
ax.set_title('Acc: raw pool vs calibrated'); ax.set_xlabel('Fraction accepted')
ax.set_ylabel('Accuracy'); ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Effect of pool calibration on FDR/Acc', fontsize=13)
plt.tight_layout(); plt.show()


### MAE summary — raw strategies

In [ ]:
print('='*60)
print('MAE SUMMARY — Raw Decoy Strategies')
print('='*60)
print(f"  {'Strategy':<25} {'err_ST':>8}  {'err_TA':>8}")
print(f"  {'-'*45}")
for strat_name, _ in STRATEGIES:
    c = raw_results[strat_name]
    print(f"  {STRATEGY_LABELS[strat_name]:<25} {c['err_st_mm']:>8.4f}  {c['err_ta_mm']:>8.4f}")
if 'curves_cal' in dir():
    print(f"  {'Calibrated (SC)':<25} {curves_cal['err_st_mm']:>8.4f}  {curves_cal['err_ta_mm']:>8.4f}")

### Deep dive: `binary_coord` strategy

Detailed FDR/Acc analysis for the `binary_coord` strategy — the one using the proper binary null pool `P(score_c | label ≠ c)` instead of the error-conditioned pool.

In [ ]:
# ── binary_coord: detailed FDR + Acc + scatter ────────────────────────────────
bc = raw_results['binary_coord']
sc = raw_results['score_coord']
r  = bc['normalized_rank']

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# (0,0) FDR: binary_coord vs score_coord vs true
ax = axes[0][0]
ax.plot(r, bc['QVAL_true'],   'k--', lw=2.5, label='True FDR')
ax.plot(r, bc['QVAL_mixmax'], color='#9C27B0', lw=2,   label=f'Binary pool  err={bc["err_st_mm"]:.3f}')
ax.plot(r, sc['QVAL_mixmax'], color='#1976D2', lw=1.5, ls='-.', label=f'Old pool  err={sc["err_st_mm"]:.3f}')
ax.plot(r, bc['QVAL_TDC'],   color='navy', lw=1, ls=':', label='TDC (binary)')
ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
ax.set_ylabel('q-value'); ax.set_xlabel('Fraction accepted')
ax.set_title('FDR: binary pool vs old pool'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)
ax.set_ylim(0, 1.05)

# (0,1) Accuracy
ax = axes[0][1]
ax.plot(r, bc['Acc_true'],   'k--', lw=2.5, label='True Acc')
ax.plot(r, bc['Acc_est_MM'], color='#9C27B0', lw=2,   label=f'Binary pool  err={bc["err_ta_mm"]:.3f}')
ax.plot(r, sc['Acc_est_MM'], color='#1976D2', lw=1.5, ls='-.', label=f'Old pool  err={sc["err_ta_mm"]:.3f}')
ax.set_ylabel('Accuracy'); ax.set_xlabel('Fraction accepted')
ax.set_title('Accuracy: binary pool vs old pool'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (0,2) Score distributions
ax = axes[0][2]
bins_d = np.linspace(bc['pred_scores'].min()-0.3, bc['pred_scores'].max()+0.3, 60)
sns.histplot(bc['pred_scores'],  bins=bins_d, stat='density', color='steelblue',
             kde=True, fill=True, alpha=0.3, label='model', ax=ax)
sns.histplot(bc['decoy_scores'], bins=bins_d, stat='density', color='#9C27B0',
             kde=True, fill=True, alpha=0.4, label='decoy (binary)', ax=ax)
sns.histplot(sc['decoy_scores'], bins=bins_d, stat='density', color='#1976D2',
             kde=True, fill=True, alpha=0.2, label='decoy (old)', ax=ax)
ax.set_title('Score distributions'); ax.set_xlabel('Max logit')
ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (1,0) Scatter: max(model) vs max(decoy) — binary_coord
ax = axes[1][0]
ms_bc = bc['pred_scores']; ds_bc = bc['decoy_scores']
corr = bc['correct'].astype(bool)
idx = np.random.default_rng(0).choice(len(ms_bc), min(4000, len(ms_bc)), replace=False)
ax.scatter(ms_bc[idx][corr[idx]],  ds_bc[idx][corr[idx]],  s=5, alpha=0.3, color='steelblue',
           label='correct', rasterized=True)
ax.scatter(ms_bc[idx][~corr[idx]], ds_bc[idx][~corr[idx]], s=5, alpha=0.5, color='crimson',
           label='incorrect', rasterized=True)
lims = [min(ms_bc.min(), ds_bc.min())-0.2, max(ms_bc.max(), ds_bc.max())+0.2]
ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel('max(model)'); ax.set_ylabel('max(decoy)')
ax.set_title('Binary pool: max(model) vs max(decoy)')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,1) Margin analysis for binary_coord — does pool still get masked by top2?
dc_bc_full = apply_strategy(sc_np, pool_score, pool_error_vectors, 'binary_coord', 0.0,
                            binary_pool=binary_pool_score)
sorted_l = np.sort(sc_np, axis=1)[:, ::-1]
top2_all = sorted_l[:, 1]
dc_at_c_bc = np.array([dc_bc_full[j, sc_np[j].argmax()] for j in range(len(sc_np))])
pool_relevant_bc = dc_at_c_bc > top2_all
corr_m = sc_np.argmax(1) == lb_np

ax = axes[1][1]
ax.bar([0, 1, 2],
       [pool_relevant_bc.mean()*100, pool_relevant_bc[corr_m].mean()*100, pool_relevant_bc[~corr_m].mean()*100],
       color=['gray', 'steelblue', 'crimson'], alpha=0.7)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['All', 'Correct', 'Incorrect'])
ax.set_ylabel('% pool draw > top2')
ax.set_title(f'Binary pool relevance\n(pool controls max(decoy): {pool_relevant_bc.mean()*100:.0f}%)')
ax.grid(ls='--', alpha=0.3)

# Also compare with score_coord pool relevance
dc_sc_full2 = apply_strategy(sc_np, pool_score, pool_error_vectors, 'score_coord', 0.0,
                             binary_pool=binary_pool_score)
dc_at_c_sc = np.array([dc_sc_full2[j, sc_np[j].argmax()] for j in range(len(sc_np))])
pool_rel_sc = dc_at_c_sc > top2_all
for x, lbl, val in [(0,'All',pool_rel_sc.mean()), (1,'Corr',pool_rel_sc[corr_m].mean()),
                     (2,'Inc',pool_rel_sc[~corr_m].mean())]:
    ax.bar(x, val*100, color='orange', alpha=0.3, width=0.5)
ax.legend(['Binary pool', 'Old pool'], fontsize=8, loc='upper right')

# (1,2) MAE comparison bar chart
ax = axes[1][2]
strats_show = ['score_coord', 'binary_coord', 'full_vector', 'nearest_neighbor']
x = np.arange(len(strats_show))
st_vals = [raw_results[s]['err_st_mm'] for s in strats_show]
ta_vals = [raw_results[s]['err_ta_mm'] for s in strats_show]
ax.bar(x - 0.2, st_vals, 0.35, color='#E53935', alpha=0.7, label='err_ST')
ax.bar(x + 0.2, ta_vals, 0.35, color='#1976D2', alpha=0.7, label='err_TA')
ax.set_xticks(x); ax.set_xticklabels([STRATEGY_LABELS[s] for s in strats_show], fontsize=8, rotation=15)
ax.set_ylabel('Absolute error'); ax.set_title('MAE comparison (key strategies)')
ax.legend(fontsize=9); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — binary_coord strategy deep dive', fontsize=14)
plt.tight_layout(); plt.show()

print(f"\n  {'Strategy':<25} {'err_ST':>8}  {'err_TA':>8}")
print(f"  {'-'*45}")
for s in strats_show:
    c = raw_results[s]
    print(f"  {STRATEGY_LABELS[s]:<25} {c['err_st_mm']:>8.4f}  {c['err_ta_mm']:>8.4f}")


### Per-class binary FDR from multiclass decoys (no retraining)

Take the multiclass `binary_coord` decoys (already computed above) and evaluate FDR as a **binary classifier** for one class.

- Score = `logit[:, c]` (not `max(logits)`)
- Decoy = `decoy[:, c]` (the c-th coordinate of the multiclass decoy vector)
- Positive = `label == c`
- All 31112 samples

This shows whether the multiclass decoy at coordinate c behaves like a proper null for the binary test "is this class c?"

In [ ]:
TARGET_C = 204

# Use the multiclass binary_coord decoys (already computed in raw_results loop)
dc_mc = apply_strategy(sc_np, pool_score, pool_error_vectors, 'binary_coord', 0.0,
                       binary_pool=binary_pool_score)

model_at_c = sc_np[:, TARGET_C]
decoy_at_c = dc_mc[:, TARGET_C]
is_pos     = (lb_np == TARGET_C)
n = len(model_at_c)

print(f"Per-class binary FDR from multiclass decoys — class {TARGET_C}")
print(f"  n={n}  pos={is_pos.sum()}  neg={(~is_pos).sum()}")
print(f"  model logit at {TARGET_C}: pos median={np.median(model_at_c[is_pos]):.2f}  "
      f"neg median={np.median(model_at_c[~is_pos]):.2f}")
print(f"  decoy at {TARGET_C}: pos median={np.median(decoy_at_c[is_pos]):.2f}  "
      f"neg median={np.median(decoy_at_c[~is_pos]):.2f}")

# p-values from decoy at coord c
# For binary_coord: decoy[i, c_hat] = pool draw, decoy[i, j≠c_hat] = model[i, j]
# So decoy_at_c depends on whether pred == c or not:
#   if pred_i == TARGET_C: decoy_at_c = pool draw from binary_pool[TARGET_C]
#   if pred_i != TARGET_C: decoy_at_c = model_at_c (unchanged!)
pred = sc_np.argmax(1)
pred_is_c = pred == TARGET_C
print(f"  samples predicted as {TARGET_C}: {pred_is_c.sum()} ({pred_is_c.mean()*100:.1f}%)")
print(f"  → for {(~pred_is_c).sum()} samples decoy_at_{TARGET_C} = model_at_{TARGET_C} (unchanged)")

# Sort by model_at_c descending
sidx = np.argsort(model_at_c)
ms_s = model_at_c[sidx]
cs_s = is_pos[sidx].astype(int)
ds_s = decoy_at_c[sidx]
DC = np.arange(n, 0, -1)

# True FDR
FD_true = np.cumsum((1 - cs_s)[::-1])[::-1]
QVAL_true = np.clip(np.minimum.accumulate(np.clip(FD_true / DC, 0, 1)), 0, 1)

# TDC (1D at coord c)
tsc = np.maximum(model_at_c, decoy_at_c)
twin = (model_at_c > decoy_at_c).astype(int)
ti = np.argsort(tsc)
FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
DC_t = np.maximum(DC - FC_t, 1)
QVAL_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)

# Mix-Max (1D at coord c)
sd = np.sort(decoy_at_c); uz, cz = np.unique(decoy_at_c, return_counts=True); nuz = len(uz)
PW = np.clip(np.searchsorted(ms_s, uz, 'left') / n, 0, 1)
PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
fdr_mm = np.zeros(n)
for i, T in enumerate(ms_s[::-1]):
    D = i + 1; zi = np.searchsorted(uz, T, 'left')
    fdr_mm[i] = (0.0 if zi >= nuz else (Rj[zi:] * cz[zi:]).sum()) / D if D > 0 else 0
QVAL_mm = np.clip(np.minimum.accumulate(np.clip(fdr_mm, 0, 1)[::-1]), 0, 1)

# Also BH with binary pool directly
pool_c = binary_pool_score[TARGET_C]
pool_sorted = np.sort(pool_c)
pv = 1.0 - np.searchsorted(pool_sorted, model_at_c, side='right') / len(pool_sorted)
pv = np.clip(pv, 1.0 / len(pool_sorted), 1.0)
si_bh = np.argsort(pv)
qv_bh = np.zeros(n)
qv_sorted = np.minimum.accumulate((pv[si_bh] * n / np.arange(1, n + 1))[::-1])[::-1]
qv_bh[si_bh] = np.clip(qv_sorted, 0, 1)
bh_by_score = qv_bh[sidx]

r = np.arange(n) / n

# ── PLOTS ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Scatter all 31112: model_at_c vs decoy_at_c
ax = axes[0][0]
ax.scatter(model_at_c[~is_pos], decoy_at_c[~is_pos],
           s=2, alpha=0.1, color='gray', label=f'neg ({(~is_pos).sum()})', rasterized=True)
ax.scatter(model_at_c[is_pos],  decoy_at_c[is_pos],
           s=8, alpha=0.6, color='crimson', label=f'pos ({is_pos.sum()})', rasterized=True)
lo = min(model_at_c.min(), decoy_at_c.min()) - 0.5
hi = max(model_at_c.max(), decoy_at_c.max()) + 0.5
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel(f'Model logit at {TARGET_C}'); ax.set_ylabel(f'Decoy logit at {TARGET_C}')
ax.set_title(f'Multiclass decoy at coord {TARGET_C}\n'
             f'(only pred={TARGET_C} get replaced, rest unchanged)')
ax.legend(fontsize=8, markerscale=2); ax.grid(ls='--', alpha=0.3)

# (0,1) Score distributions
ax = axes[0][1]
bins_b = np.linspace(lo, hi, 80)
ax.hist(model_at_c[is_pos],  bins=bins_b, density=True, alpha=0.5, color='crimson', label=f'pos model')
ax.hist(model_at_c[~is_pos], bins=bins_b, density=True, alpha=0.2, color='gray', label='neg model')
ax.hist(decoy_at_c[pred_is_c], bins=bins_b, density=True, histtype='step', lw=2, color='#9C27B0',
        label=f'decoy (pred={TARGET_C}, replaced)')
ax.hist(pool_c, bins=bins_b, density=True, histtype='step', lw=1.5, ls='--', color='orange',
        label=f'binary pool')
ax.set_xlabel(f'Logit at {TARGET_C}'); ax.set_ylabel('Density')
ax.set_title('Distributions at coord c'); ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (0,2) p-value histograms
ax = axes[0][2]
bins_p = np.linspace(0, 1, 31)
ax.hist(pv[~is_pos], bins=bins_p, density=True, alpha=0.5, color='#2E7D32', label=f'Neg (mean={pv[~is_pos].mean():.2f})')
ax.hist(pv[is_pos],  bins=bins_p, density=True, alpha=0.5, color='crimson', label=f'Pos (mean={pv[is_pos].mean():.2f})')
ax.axhline(1, ls='--', color='black', lw=1)
ax.set_xlabel('p-value'); ax.set_ylabel('Density')
ax.set_title('BH p-values (from binary pool directly)'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,0) FDR curves
ax = axes[1][0]
ax.plot(r, QVAL_true, 'k--', lw=2.5, label='True FDR')
ax.plot(r, QVAL_mm,   color='#9C27B0', lw=2,   label='MixMax (from multiclass decoy)')
ax.plot(r, QVAL_TDC,  color='navy',    lw=1.2, ls=':', label='TDC')
ax.plot(r, bh_by_score, color='#2E7D32', lw=1.5, ls='-.', label='BH (from binary pool)')
ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value')
ax.set_ylim(0, 1.05)
ax.set_title(f'Binary FDR — class {TARGET_C}\nfrom multiclass binary_coord decoys')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,1) Zoom top 10%
ax = axes[1][1]
zoom = r > 0.9
ax.plot(r[zoom], QVAL_true[zoom],    'k--', lw=2.5, label='True FDR')
ax.plot(r[zoom], QVAL_mm[zoom],      color='#9C27B0', lw=2,   label='MixMax (multiclass decoy)')
ax.plot(r[zoom], bh_by_score[zoom],  color='#2E7D32', lw=1.5, ls='-.', label='BH (binary pool)')
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value'); ax.set_ylim(0, 1.05)
ax.set_title('ZOOM: top 10%'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,2) Precision-Recall
ax = axes[1][2]
tp_cum = np.cumsum(cs_s[::-1])[::-1]
prec_true = tp_cum / DC; rec_true = tp_cum / max(is_pos.sum(), 1)
prec_mm = np.clip(1 - QVAL_mm, 0, 1); rec_mm = prec_mm * DC / max(is_pos.sum(), 1)
prec_bh = np.clip(1 - bh_by_score, 0, 1); rec_bh = prec_bh * DC / max(is_pos.sum(), 1)
ax.plot(rec_true, prec_true, 'k--', lw=2.5, label='True PR')
ax.plot(rec_mm, prec_mm, color='#9C27B0', lw=2, label='MixMax (multiclass)')
ax.plot(rec_bh, prec_bh, color='#2E7D32', lw=1.5, ls='-.', label='BH (binary pool)')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.set_title(f'PR — class {TARGET_C}'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

plt.suptitle(f'Binary FDR from multiclass binary_coord decoys — class {TARGET_C}\n'
             f'n={n}  pos={is_pos.sum()}  (no retraining, just slice coord {TARGET_C})',
             fontsize=13)
plt.tight_layout(); plt.show()

# Summary
print(f"\nDiscoveries at controlled FDR (class {TARGET_C}, binary from multiclass decoys):")
print(f"  {'alpha':<7} {'BH(pool)':>12} {'MM(decoy)':>12} {'TDC':>12}  (/{is_pos.sum()} pos)")
print(f"  {'-'*52}")
for alpha in [0.01, 0.05, 0.10, 0.20]:
    bh_d = (qv_bh <= alpha).sum(); bh_tp = (qv_bh[is_pos] <= alpha).sum()
    mm_m = QVAL_mm <= alpha; mm_d = mm_m.sum(); mm_tp = cs_s[mm_m].sum()
    td_m = QVAL_TDC <= alpha; td_d = td_m.sum(); td_tp = cs_s[td_m].sum() if td_d else 0
    print(f"  {alpha:<7.2f} {bh_d:>5}({bh_tp:>3}tp) {mm_d:>5}({mm_tp:>3}tp) {td_d:>5}({td_tp:>3}tp)")


## Flow training — all 4 strategies

Trains a separate normalizing flow for each decoy strategy. Saves checkpoints to `plant_flow_{strategy}_half.pth`.

In [ ]:
FLOW_EPOCHS    = 30
FLOW_LR        = 3e-4
FLOW_PATIENCE  = 5
FLOW_N         = 12
FLOW_ENC_DIM   = 128
FLOW_SUBSAMPLE = 0.5
FLOW_SEED      = 42

# Which strategies to train flows for (subset to save time)
FLOW_STRATEGIES = [
    ('score_coord',    0.0),
    ('binary_coord',   0.0),
    ('full_vector',    0.0),
]

n_total = len(train_scores_raw)
n_flow  = int(n_total * FLOW_SUBSAMPLE)
sub_idx = np.random.default_rng(FLOW_SEED).choice(n_total, size=n_flow, replace=False)
sub_idx.sort()

flow_tr_sc = train_scores_raw[sub_idx]
flow_tr_ft = train_features.numpy()[sub_idx]
flow_tr_lb = train_labels_raw[sub_idx]
print(f"Flow subset: {n_flow}/{n_total} ({FLOW_SUBSAMPLE*100:.0f}%)"
      f"  acc={(flow_tr_sc.argmax(1)==flow_tr_lb).mean():.4f}")

flow_results = {}
for strat_name, noise_std in FLOW_STRATEGIES:
    print(f'\n{"#"*55}  {strat_name}  (noise={noise_std})')
    train_decoy = apply_strategy(flow_tr_sc, pool_score, pool_error_vectors, strat_name, noise_std,
                                 binary_pool=binary_pool_score)
    train_ds = ScoreFeatureDataset(
        torch.from_numpy(flow_tr_sc).float(), torch.from_numpy(flow_tr_ft).float(),
        torch.from_numpy(train_decoy).float(), torch.from_numpy(flow_tr_lb).long())

    flow_path = f'plant_flow_{strat_name}_half.pth'
    flow = ScoreShiftFlowWrapper(NUM_CLASSES, FLOW_N, FEATURE_DIM, 256, FLOW_ENC_DIM, 5.0).to(DEVICE)

    if os.path.exists(flow_path):
        flow.load_state_dict(torch.load(flow_path, map_location=DEVICE, weights_only=False))
        print(f"  Loaded from {flow_path}")
    else:
        print(f"  Training -> {flow_path}")
        flow.train_flow(train_ds, epochs=FLOW_EPOCHS, lr=FLOW_LR,
                        batch_size=256, device=str(DEVICE),
                        patience=FLOW_PATIENCE, grad_clip=1.0)
        torch.save(flow.state_dict(), flow_path)
        print(f"  Saved -> {flow_path}")

    flow.eval()
    test_ds = ScoreFeatureDataset(
        torch.from_numpy(sc_np).float(), torch.from_numpy(ft_np).float(),
        torch.from_numpy(sc_np).float(),  # placeholder, ignored by generate_decoys
        torch.from_numpy(lb_np).long())
    ms_np, ds_np, ls_np = flow.generate_decoys(test_ds, device=str(DEVICE))
    crv = compute_fdr_acc_curves(ms_np, ds_np, ls_np)
    flow_results[strat_name] = crv
    print(f"  true_acc={crv['true_acc']:.3f}  err_st={crv['err_st_mm']:.3f}  err_ta={crv['err_ta_mm']:.3f}")

print('\nFlow experiments done.')

### Raw vs Flow — FDR and Accuracy curves

In [ ]:
r = raw_results['score_coord']['normalized_rank']
n_fs = len(FLOW_STRATEGIES)
fig, axes = plt.subplots(2, n_fs, figsize=(5*n_fs, 8))
flow_strat_names = [s for s, _ in FLOW_STRATEGIES]
for col, strat_name in enumerate(flow_strat_names):
    rc = raw_results[strat_name]; fc = flow_results[strat_name]; color = STRATEGY_COLORS[strat_name]

    ax = axes[0][col]
    ax.plot(r, rc['QVAL_true'],    color='black',      lw=2,   ls='--', label='True FDR')
    ax.plot(r, rc['QVAL_mixmax'],  color=color,        lw=1.8,          label=f'Raw  {rc["err_st_mm"]:.3f}')
    ax.plot(r, fc['QVAL_mixmax'],  color='darkorange', lw=1.8, ls='-.', label=f'Flow {fc["err_st_mm"]:.3f}')
    ax.set_title(STRATEGY_LABELS[strat_name]); ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
    if col == 0: ax.set_ylabel('q-value (FDR)')

    ax = axes[1][col]
    ax.plot(r, rc['Acc_true'],     color='black',      lw=2,   ls='--', label='True Acc')
    ax.plot(r, rc['Acc_est_MM'],   color=color,        lw=1.8,          label=f'Raw  {rc["err_ta_mm"]:.3f}')
    ax.plot(r, fc['Acc_est_MM'],   color='darkorange', lw=1.8, ls='-.', label=f'Flow {fc["err_ta_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted'); ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)
    if col == 0: ax.set_ylabel('Accuracy')

plt.suptitle('Plant — Raw vs Flow decoys (all strategies)', fontsize=13)
plt.tight_layout(); plt.show()


### Raw vs Flow — score distributions

In [ ]:
ref = raw_results['score_coord']
bins = np.linspace(ref['pred_scores'].min() - 0.3, ref['pred_scores'].max() + 0.3, 60)
n_fs = len(FLOW_STRATEGIES)
fig, axes = plt.subplots(2, n_fs, figsize=(5*n_fs, 8))
flow_strat_names = [s for s, _ in FLOW_STRATEGIES]
for col, strat_name in enumerate(flow_strat_names):
    rc = raw_results[strat_name]; fc = flow_results[strat_name]; color = STRATEGY_COLORS[strat_name]
    for row, (c, lbl, col2) in enumerate([(rc, 'RAW', color), (fc, 'FLOW', 'darkorange')]):
        ax = axes[row][col]
        sns.histplot(c['pred_scores'],  bins=bins, stat='density', color='steelblue',
                     kde=True, fill=True, alpha=0.3, label='model', ax=ax)
        sns.histplot(c['decoy_scores'], bins=bins, stat='density', color=col2,
                     kde=True, fill=True, alpha=0.4, label='decoy', ax=ax)
        ax.set_title(f'{STRATEGY_LABELS[strat_name]}  {lbl}  err_st={c["err_st_mm"]:.3f}')
        ax.set_xlabel('Max logit'); ax.legend(fontsize=7)
        if col == 0: ax.set_ylabel(f'Density ({lbl.lower()})')
plt.suptitle('Plant — Raw vs Flow score distributions', fontsize=13)
plt.tight_layout(); plt.show()


### Final MAE summary — Raw vs Flow

In [ ]:
print('='*70)
print('MAE SUMMARY — Raw vs Flow (all strategies)')
print('='*70)
print(f"  {'Strategy':<25} {'Raw ST':>8}  {'Raw TA':>8}  {'Flow ST':>8}  {'Flow TA':>8}")
print(f"  {'-'*65}")
for strat_name, _ in FLOW_STRATEGIES:
    rc = raw_results[strat_name]; fc = flow_results[strat_name]
    print(f"  {STRATEGY_LABELS[strat_name]:<25} "
          f"{rc['err_st_mm']:>8.4f}  {rc['err_ta_mm']:>8.4f}  "
          f"{fc['err_st_mm']:>8.4f}  {fc['err_ta_mm']:>8.4f}")
print(f"  {'Calibrated pool (SC)':<25} {curves_cal['err_st_mm']:>8.4f}  {curves_cal['err_ta_mm']:>8.4f}")
